### using python kafka  kafka 

In [ ]:
from kafka import KafkaConsumer, TopicPartition, OffsetAndMetadata
from kafka.errors import KafkaError
import json
import logging

logging.basicConfig(level=logging.INFO)

# Create a Kafka consumer
consumer = KafkaConsumer(
    'example_topic',
    bootstrap_servers='localhost:9092',
    group_id='my-python-group',
    enable_auto_commit=False,  # Manual commit
    auto_offset_reset='earliest',  # Start from beginning if no offset
    value_deserializer=lambda m: m.decode('utf-8')
)

print("Consumer started...")

try:
    while True:
        # Poll for messages
        records = consumer.poll(timeout_ms=1000)

        for topic_partition, messages in records.items():
            for message in messages:
                print(f"Received: {message.value} | Offset: {message.offset}")

                # Process the message (put your logic here)
                # ...

            # ✅ Manual Synchronous Commit (reliable)
            try:
                consumer.commit()
                print(f"✔ Sync Commit done for partition {topic_partition}")
            except KafkaError as e:
                logging.error(f"❌ Sync Commit failed: {e}")

            # ✅ Manual Asynchronous Commit (faster, with callback)
            def on_commit_success(offsets, exception):
                if exception:
                    logging.error(f"❌ Async Commit failed: {exception}")
                else:
                    logging.info(f"✔ Async Commit successful: {offsets}")

            consumer.commit_async(callback=on_commit_success)

except KeyboardInterrupt:
    print("Stopping consumer...")

finally:
    try:
        # Do a final sync commit before exit for safety
        consumer.commit()
    except KafkaError as e:
        logging.error(f"Final commit failed: {e}")
    consumer.close()
    print("Consumer closed.")


In [ ]:
from confluent_kafka import Consumer, KafkaError
from confluent_kafka.schema_registry import SchemaRegistryClient
from confluent_kafka.schema_registry.avro import AvroDeserializer
from confluent_kafka.serialization import SerializationContext, MessageField
import logging

logging.basicConfig(level=logging.INFO)

# === Step 1: Configuration ===

# Kafka and Schema Registry credentials (replace with your real credentials)
conf = {
    'bootstrap.servers': 'your-cluster.kafka.confluent.cloud:9092',
    'group.id': 'my-python-group',
    'auto.offset.reset': 'earliest',
    'enable.auto.commit': False,
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': 'your_kafka_api_key',
    'sasl.password': 'your_kafka_api_secret'
}

schema_registry_conf = {
    'url': 'https://your-schema-registry-url',
    'basic.auth.user.info': 'your_schema_registry_api_key:your_schema_registry_api_secret'
}

# === Step 2: Set up schema registry and deserializer ===

schema_registry_client = SchemaRegistryClient(schema_registry_conf)

# Avro Deserializer (replace with JSON/Protobuf deserializer if needed)
avro_deserializer = AvroDeserializer(
    schema_registry_client=schema_registry_client,
    schema_str=None  # Will auto-fetch latest schema for the topic
)

# === Step 3: Create Consumer ===
consumer = Consumer(conf)
consumer.subscribe(['example_topic'])

print("Confluent Kafka Consumer started...")

try:
    while True:
        msg = consumer.poll(1.0)
        if msg is None:
            continue
        if msg.error():
            print(f"Kafka Error: {msg.error()}")
            continue

        # Deserialize using Avro
        record_value = avro_deserializer(msg.value(), SerializationContext(msg.topic(), MessageField.VALUE))
        print(f"Consumed record from topic {msg.topic()} at offset {msg.offset()}: {record_value}")

        # Process message (your logic here)
        # ...

        # Manual Sync Commit
        try:
            consumer.commit()  # Blocking commit
            print("✔ Sync commit successful")
        except KafkaError as e:
            logging.error(f"❌ Sync Commit Failed: {e}")

        # Async Commit with callback
        def on_commit(err, partitions):
            if err:
                logging.error(f"❌ Async Commit failed: {err}")
            else:
                logging.info(f"✔ Async Commit successful: {partitions}")

        consumer.commit(asynchronous=True, callback=on_commit)

except KeyboardInterrupt:
    print("Stopping consumer...")

finally:
    try:
        consumer.commit()  # Final commit
    except KafkaError as e:
        logging.error(f"Final commit failed: {e}")
    consumer.close()
    print("Consumer closed.")
